In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import math
import pickle

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# from fix_data_leakage import scale_data_without_leakage, create_sequences
from sklearn.preprocessing import StandardScaler


In [16]:
!nvidia-smi

Tue Mar  3 16:20:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/HoangHumg1210/hoankiem-air-quality-.git
%cd hoankiem-air-quality-
!ls

fatal: destination path 'hoankiem-air-quality-' already exists and is not an empty directory.
/content/hoankiem-air-quality-
data  notebooks  requirements.txt


In [12]:
df = pd.read_csv('data/processed/data2225_clean.csv')
df.head()

,Local Time,index,CO,NO2,O3,PM10,PM25,SO2,Clouds,Precipitation,Pressure,Relative Humidity,Temperature,UV Index,Wind Speed,HolidayName,IsHoliday,Accumulated Hours of Rain
0,2022-01-13 07:00:00,0,353.1,10.0,84.0,98.0,17.08,52.0,100,0.00,1020.0,95.0,15.5,0.6,2.00,Ngày thường,False,0
1,2022-01-13 08:00:00,1,343.5,9.0,87.3,95.7,16.75,48.7,91,0.00,1021.0,94.0,15.4,0.7,2.33,Ngày thường,False,0
2,2022-01-13 09:00:00,2,334.0,8.0,90.7,93.3,16.42,45.3,83,0.50,1022.0,93.0,15.3,1.0,2.66,Ngày thường,False,1
3,2022-01-13 10:00:00,3,324.5,7.0,94.0,91.0,16.09,42.0,75,0.75,1022.0,93.0,15.2,1.5,3.00,Ngày thường,False,2
4,2022-01-13 11:00:00,4,319.6,6.7,95.7,91.3,16.17,39.0,83,0.00,1021.0,87.0,15.6,1.9,3.00,Ngày thường,False,0


In [14]:
df.drop(columns=['index'], inplace=True)
df['Local Time'] = pd.to_datetime(df['Local Time'])

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34759 entries, 0 to 34758
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Local Time                 34759 non-null  object 
 1   index                      34759 non-null  int64  
 2   CO                         34759 non-null  float64
 3   NO2                        34759 non-null  float64
 4   O3                         34759 non-null  float64
 5   PM10                       34759 non-null  float64
 6   PM25                       34759 non-null  float64
 7   SO2                        34759 non-null  float64
 8   Clouds                     34759 non-null  int64  
 9   Precipitation              34759 non-null  float64
 10  Pressure                   34759 non-null  float64
 11  Relative Humidity          34759 non-null  float64
 12  Temperature                34759 non-null  float64
 13  UV Index                   34759 non-null  flo

In [15]:
df = df.set_index('Local Time').sort_index()


In [ ]:
idx = df_daily.index
df_daily['day_of_week'] = idx.dayofweek
df_daily['month'] = idx.month
df_daily['hour_sin'] = np.sin(2*np.pi*idx.hour/24)
df_daily['hour_cos'] = np.cos(2*np.pi*idx.hour/24)
df_daily['is_weekend'] = (idx.dayofweek >= 5).astype(int)


In [ ]:
target = 'PM25'
base_features = df_daily.columns.drop(target)

print("n_base_features:", len(base_features))
print(base_features)


In [ ]:
lags = [1, 2, 4, 8, 16] 


for l in lags:
    df_daily[f"{target}_lag{l}"] = df_daily[target].shift(l)

df_daily = df_daily.interpolate(method='time') 


In [ ]:
train_end = "2023-12-31"
val_start = "2024-01-01"
val_end = "2024-12-31"
test_start = "2025-01-01"

train_df = df_daily[:train_end]
val_df = df_daily[val_start:val_end]
test_df = df_daily[test_start:]

print(train_df.shape, val_df.shape, test_df.shape)
assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0, "One of train/val/test splits is empty."


In [ ]:
lag_features = [f"{target}_lag{l}" for l in lags]

features_with_lags = list(dict.fromkeys(base_features.tolist() + lag_features))

print("Features with lags:", features_with_lags)


In [ ]:

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    numeric_values = split_df.select_dtypes(include=[np.number]).values
    assert np.isfinite(numeric_values).all(), f"{split_name} split contains non-finite numeric values."


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(train_df.index, train_df[target], label="Train")
plt.plot(val_df.index, val_df[target], label="Val")
plt.plot(test_df.index, test_df[target], label="Test")
plt.title("PM2.5 - Train/Val/Test split theo ")
plt.legend()


In [ ]:
min_corr = 0.02
exclude_high_corr = None

corr = train_df[features_with_lags].corrwith(train_df[target]).abs()
selected_corr = corr[corr >= min_corr].index.tolist()

if exclude_high_corr is not None:
    too_similar = corr[corr > exclude_high_corr].index.tolist()
    selected_corr = [f for f in selected_corr if f not in too_similar]

print(f"Correlation filter: {len(selected_corr)}/{len(features_with_lags)} selected")
print(selected_corr)


In [ ]:
tree_top_k = 12
random_state = 42
n_estimators = 200

rf = RandomForestRegressor(
    n_estimators=n_estimators,
    random_state=random_state,
    n_jobs=1
)

rf.fit(train_df[features_with_lags], train_df[target])

importances = pd.Series(rf.feature_importances_, index=features_with_lags).sort_values(ascending=False)
selected_tree = importances.head(tree_top_k).index.tolist()

print(selected_tree)
print("\nCÃƒÂ¡c Ã„â€˜Ã¡ÂºÂ·c trÃ†Â°ng cÃƒÂ³ Ã„â€˜Ã¡Â»â„¢ quan trÃ¡Â»Âng cao nhÃ¡ÂºÂ¥t:")
print(importances.head(tree_top_k))


In [ ]:
def scale_data_without_leakage(train, val, test, feature_range=None):
    if feature_range is None:
        # StandardScaler (mean=0, std=1)
        scaler = StandardScaler()
        print("Using StandardScaler (z-score normalization)")
    else:
        # MinMaxScaler
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler(feature_range=feature_range)
        print(f"Using MinMaxScaler (range: {feature_range})")

    # CRITICAL: Fit ONLY on training data
    scaler.fit(train)

    # Transform all sets using SAME scaler (train statistics)
    train_scaled = scaler.transform(train)
    val_scaled = scaler.transform(val)
    test_scaled = scaler.transform(test)

    print("âœ“ Scaling completed without data leakage")

    return train_scaled, val_scaled, test_scaled, scaler


USE_LOG_TARGET = True


def forward_target_transform(y, use_log=USE_LOG_TARGET):
    y_arr = np.asarray(y, dtype=np.float64)
    if use_log:
        if np.any(y_arr < 0):
            raise ValueError("Target contains negative values; cannot apply log1p safely.")
        return np.log1p(y_arr)
    return y_arr


def inverse_target_transform(y, use_log=USE_LOG_TARGET):
    y_arr = np.asarray(y, dtype=np.float64)
    if use_log:
        return np.maximum(np.expm1(y_arr), 0.0)
    return y_arr


def create_sequences(X, y, time_step=30):
   
    X_seq, y_seq = [], []

    for i in range(len(X) - time_step):
        X_seq.append(X[i:i + time_step])
        y_seq.append(y[i + time_step])

    return np.array(X_seq), np.array(y_seq)

In [ ]:
LOOKBACK = 24
EPOCHS = 20
BATCH_SIZE = 64

feature_sets = [
    ("all_features", features_with_lags),
    ("corr_features", selected_corr),
    ("tree_top_k", selected_tree),
]

rows = []

for set_name, cols in feature_sets:
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    np.random.seed(42)

    X_train = train_df[cols].values
    X_val   = val_df[cols].values
    X_test  = test_df[cols].values

    y_train_raw = train_df[target].values.reshape(-1, 1)
    y_val_raw   = val_df[target].values.reshape(-1, 1)
    y_test_raw  = test_df[target].values.reshape(-1, 1)

    y_train = forward_target_transform(y_train_raw, use_log=USE_LOG_TARGET)
    y_val   = forward_target_transform(y_val_raw, use_log=USE_LOG_TARGET)
    y_test  = forward_target_transform(y_test_raw, use_log=USE_LOG_TARGET)

    X_train_s, X_val_s, X_test_s, scaler_X = scale_data_without_leakage(X_train, X_val, X_test)
    y_train_s, y_val_s, y_test_s, scaler_y = scale_data_without_leakage(y_train, y_val, y_test)

    X_train_seq, y_train_seq = create_sequences(X_train_s, y_train_s, time_step=LOOKBACK)
    X_val_seq, y_val_seq     = create_sequences(X_val_s, y_val_s, time_step=LOOKBACK)
    X_test_seq, y_test_seq   = create_sequences(X_test_s, y_test_s, time_step=LOOKBACK)

    model = Sequential([
        Conv1D(32, kernel_size=3, activation='relu', padding='causal',
               input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
        MaxPooling1D(pool_size=2),
        GRU(64, return_sequences=True),
        Dropout(0.2),
        GRU(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(
        X_train_seq, y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=0
    )

    y_val_pred_s  = model.predict(X_val_seq, verbose=0)
    y_test_pred_s = model.predict(X_test_seq, verbose=0)

    y_val_true_log  = scaler_y.inverse_transform(y_val_seq)
    y_val_pred_log  = scaler_y.inverse_transform(y_val_pred_s)
    y_test_true_log = scaler_y.inverse_transform(y_test_seq)
    y_test_pred_log = scaler_y.inverse_transform(y_test_pred_s)

    y_val_true  = inverse_target_transform(y_val_true_log, use_log=USE_LOG_TARGET)
    y_val_pred  = inverse_target_transform(y_val_pred_log, use_log=USE_LOG_TARGET)
    y_test_true = inverse_target_transform(y_test_true_log, use_log=USE_LOG_TARGET)
    y_test_pred = inverse_target_transform(y_test_pred_log, use_log=USE_LOG_TARGET)

    val_rmse  = np.sqrt(mean_squared_error(y_val_true.ravel(), y_val_pred.ravel()))
    val_mae   = mean_absolute_error(y_val_true.ravel(), y_val_pred.ravel())
    test_rmse = np.sqrt(mean_squared_error(y_test_true.ravel(), y_test_pred.ravel()))
    test_mae  = mean_absolute_error(y_test_true.ravel(), y_test_pred.ravel())

    rows.append({
        "feature_set": set_name,
        "n_features": len(cols),
        "val_rmse": val_rmse,
        "val_mae": val_mae,
        "test_rmse": test_rmse,
        "test_mae": test_mae
    })

results = pd.DataFrame(rows).sort_values(["val_rmse", "val_mae"]).reset_index(drop=True)
print(results)

In [ ]:
best_set = results.loc[0, "feature_set"]
feature_map = {
    "all_features": features_with_lags,
    "corr_features": selected_corr,
    "tree_top_k": selected_tree,
}
final_features = feature_map[best_set]
best_row = results.loc[0, ["feature_set", "n_features", "val_rmse", "val_mae", "test_rmse", "test_mae"]]

print("Best feature set:", best_set)
print("Best row:", best_row.to_dict())
print("n_final_features:", len(final_features))
print("final_features:", final_features)


In [ ]:
assert len(final_features) > 0, f"Không có đặc trưng nào cho feature set đã chọn: {best_set}"

if not any(f.startswith(f"{target}_lag") for f in final_features):
    print("Cảnh báo: Không tìm thấy đặc trưng trễ PM25 trong final_features.")

print("Các đặc trưng cuối cùng dùng để huấn luyện:", final_features)
print(f"USE_LOG_TARGET={USE_LOG_TARGET} (target transform: {'log1p' if USE_LOG_TARGET else 'none'})")

X_train = train_df[final_features].values
X_val   = val_df[final_features].values
X_test  = test_df[final_features].values

y_train_raw = train_df[target].values.reshape(-1, 1)
y_val_raw   = val_df[target].values.reshape(-1, 1)
y_test_raw  = test_df[target].values.reshape(-1, 1)

y_train = forward_target_transform(y_train_raw, use_log=USE_LOG_TARGET)
y_val   = forward_target_transform(y_val_raw, use_log=USE_LOG_TARGET)
y_test  = forward_target_transform(y_test_raw, use_log=USE_LOG_TARGET)

for split_name, X_arr, y_arr in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test)
]:
    assert np.isfinite(X_arr).all(), f"X của tập {split_name} chứa giá trị không hữu hạn."
    assert np.isfinite(y_arr).all(), f"y của tập {split_name} chứa giá trị không hữu hạn."

In [ ]:
X_train_s, X_val_s, X_test_s, scaler_X = scale_data_without_leakage(X_train, X_val, X_test)

scale_target = True
if scale_target:
    y_train_s, y_val_s, y_test_s, scaler_y = scale_data_without_leakage(y_train, y_val, y_test)
else:
    y_train_s, y_val_s, y_test_s = y_train, y_val, y_test
    scaler_y = None

for split_name, X_arr, y_arr in [
    ("train", X_train_s, y_train_s),
    ("val", X_val_s, y_val_s),
    ("test", X_test_s, y_test_s)
]:
    assert np.isfinite(X_arr).all(), \
        f"X đã chuẩn hóa của tập {split_name} chứa giá trị không hữu hạn."

    assert np.isfinite(y_arr).all(), \
        f"y đã chuẩn hóa của tập {split_name} chứa giá trị không hữu hạn."

In [ ]:
LOOKBACK = 24

X_train_seq, y_train_seq = create_sequences(X_train_s, y_train_s, time_step=LOOKBACK)
X_val_seq, y_val_seq = create_sequences(X_val_s, y_val_s, time_step=LOOKBACK)
X_test_seq, y_test_seq = create_sequences(X_test_s, y_test_s, time_step=LOOKBACK)

print("Sequence shapes:")
print("Train:", X_train_seq.shape, y_train_seq.shape)
print("Val:", X_val_seq.shape, y_val_seq.shape)
print("Test:", X_test_seq.shape, y_test_seq.shape)

for split_name, X_arr, y_arr in [
    ("train", X_train_seq, y_train_seq),
    ("val", X_val_seq, y_val_seq),
    ("test", X_test_seq, y_test_seq)
]:
    assert X_arr.shape[0] > 0 and y_arr.shape[0] > 0, f"{split_name} sequence is empty."
    assert np.isfinite(X_arr).all(), f"{split_name} X_seq contains non-finite values."
    assert np.isfinite(y_arr).all(), f"{split_name} y_seq contains non-finite values."


In [ ]:
from itertools import product
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

search_space = {
    "lookback": [12, 24, 36, 48],
    "conv_filters": [16, 32, 64],
    "gru_units": [(64, 32), (96, 48), (128, 64)],
    "dropout": [0.1, 0.2, 0.3],
    "dense_units": [16, 32],
    "batch_size": [32, 64, 128],
    "learning_rate": [1e-3, 5e-4, 1e-4],
}

all_configs = [
    {
        "lookback": lb,
        "conv_filters": cf,
        "gru_1": gu[0],
        "gru_2": gu[1],
        "dropout": dr,
        "dense_units": du,
        "batch_size": bs,
        "learning_rate": lr,
    }
    for lb, cf, gu, dr, du, bs, lr in product(
        search_space["lookback"],
        search_space["conv_filters"],
        search_space["gru_units"],
        search_space["dropout"],
        search_space["dense_units"],
        search_space["batch_size"],
        search_space["learning_rate"],
    )
]

N_TRIALS = 16
MAX_EPOCHS = 120

rng = np.random.default_rng(42)
trial_indices = rng.choice(len(all_configs), size=min(N_TRIALS, len(all_configs)), replace=False)
trial_configs = [all_configs[i] for i in trial_indices]

search_rows = []
best_obj = None

def build_cnn_gru_from_cfg(cfg, input_shape):
    model = Sequential([
        tf.keras.Input(shape=input_shape),
        Conv1D(filters=cfg["conv_filters"], kernel_size=3, activation='relu', padding='causal'),
        MaxPooling1D(pool_size=2),
        GRU(cfg["gru_1"], return_sequences=True),
        Dropout(cfg["dropout"]),
        GRU(cfg["gru_2"]),
        Dropout(cfg["dropout"]),
        Dense(cfg["dense_units"], activation='relu'),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=cfg["learning_rate"]), loss='mse', metrics=['mae'])
    return model

for i, cfg in enumerate(trial_configs, start=1):
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    np.random.seed(42)

    X_train_seq_t, y_train_seq_t = create_sequences(X_train_s, y_train_s, time_step=cfg["lookback"])
    X_val_seq_t, y_val_seq_t = create_sequences(X_val_s, y_val_s, time_step=cfg["lookback"])
    X_test_seq_t, y_test_seq_t = create_sequences(X_test_s, y_test_s, time_step=cfg["lookback"])

    model = build_cnn_gru_from_cfg(cfg, (X_train_seq_t.shape[1], X_train_seq_t.shape[2]))

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    ]

    hist = model.fit(
        X_train_seq_t,
        y_train_seq_t,
        validation_data=(X_val_seq_t, y_val_seq_t),
        epochs=MAX_EPOCHS,
        batch_size=cfg["batch_size"],
        callbacks=callbacks,
        verbose=0,
    )

    y_val_pred_s = model.predict(X_val_seq_t, verbose=0)
    if scaler_y is not None:
        y_val_true = scaler_y.inverse_transform(y_val_seq_t)
        y_val_pred = scaler_y.inverse_transform(y_val_pred_s)
    else:
        y_val_true = y_val_seq_t
        y_val_pred = y_val_pred_s
    val_rmse = np.sqrt(mean_squared_error(y_val_true.ravel(), y_val_pred.ravel()))
    val_mae = mean_absolute_error(y_val_true.ravel(), y_val_pred.ravel())

    row = {
        "trial": i,
        **cfg,
        "val_rmse": val_rmse,
        "val_mae": val_mae,
        "epochs_ran": len(hist.history['loss']),
        "best_val_loss": float(np.min(hist.history['val_loss'])),
    }
    search_rows.append(row)
    print(f"Trial {i:02d}/{len(trial_configs)} | val_rmse={val_rmse:.4f} | val_mae={val_mae:.4f} | cfg={cfg}")

    if (
        best_obj is None
        or val_rmse < best_obj["val_rmse"]
        or (np.isclose(val_rmse, best_obj["val_rmse"]) and val_mae < best_obj["val_mae"])
    ):
        best_obj = {
            "weights": model.get_weights(),
            "history": hist,
            "cfg": cfg,
            "val_rmse": val_rmse,
            "val_mae": val_mae,
            "X_train_seq": X_train_seq_t,
            "y_train_seq": y_train_seq_t,
            "X_val_seq": X_val_seq_t,
            "y_val_seq": y_val_seq_t,
            "X_test_seq": X_test_seq_t,
            "y_test_seq": y_test_seq_t,
        }

search_results = pd.DataFrame(search_rows).sort_values(["val_rmse", "val_mae"]).reset_index(drop=True)
display(search_results.head(10))

best_config = best_obj["cfg"]
tf.keras.backend.clear_session()
cnn_gru_model = build_cnn_gru_from_cfg(best_config, (best_obj["X_train_seq"].shape[1], best_obj["X_train_seq"].shape[2]))
cnn_gru_model.set_weights(best_obj["weights"])
history = best_obj["history"]

X_train_seq = best_obj["X_train_seq"]
y_train_seq = best_obj["y_train_seq"]
X_val_seq = best_obj["X_val_seq"]
y_val_seq = best_obj["y_val_seq"]
X_test_seq = best_obj["X_test_seq"]
y_test_seq = best_obj["y_test_seq"]

print("\nBest config (validation-first):", best_config)
print(f"Best val_rmse: {best_obj['val_rmse']:.4f}")
print(f"Best val_mae: {best_obj['val_mae']:.4f}")
print("Random search completed.")


In [ ]:
assert len(search_results) > 0, "search_results đang rỗng. Hãy chạy random search trước."
assert "epochs_ran" in search_results.columns, "Thiếu cột 'epochs_ran' trong search_results."
best_epochs = int(search_results.loc[0, "epochs_ran"])

X_train_val = np.vstack([X_train, X_val])
y_train_val = np.vstack([y_train, y_val])

X_train_val_s, _, X_test_final_s, scaler_X_final = scale_data_without_leakage(
    X_train_val, X_train_val, X_test
)

if scaler_y is not None:
    y_train_val_s, _, y_test_final_s, scaler_y_final = scale_data_without_leakage(
        y_train_val, y_train_val, y_test
    )
else:
    y_train_val_s = y_train_val
    y_test_final_s = y_test
    scaler_y_final = None

lb = best_config["lookback"]
X_train_val_seq, y_train_val_seq = create_sequences(X_train_val_s, y_train_val_s, time_step=lb)
X_test_final_seq, y_test_final_seq = create_sequences(X_test_final_s, y_test_final_s, time_step=lb)

assert X_train_val_seq.shape[0] > 0, "Tập sequence train+val đang rỗng."
assert X_test_final_seq.shape[0] > 0, "Tập sequence test đang rỗng."

tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

cnn_gru_model_final = build_cnn_gru_from_cfg(
    best_config,
    (X_train_val_seq.shape[1], X_train_val_seq.shape[2])
)

history_final = cnn_gru_model_final.fit(
    X_train_val_seq,
    y_train_val_seq,
    epochs=best_epochs,
    batch_size=best_config["batch_size"],
    shuffle=False,
    verbose=1
)

cnn_gru_model = cnn_gru_model_final
X_test_seq = X_test_final_seq
y_test_seq = y_test_final_seq
scaler_y = scaler_y_final

print("Huấn luyện lại mô hình cuối cùng trên tập train+val đã hoàn tất.")
print("Cấu hình tốt nhất:", best_config)
print("Số epoch tốt nhất:", best_epochs)
print("Kích thước sequence train+val:", X_train_val_seq.shape, y_train_val_seq.shape)
print("Kích thước sequence test:", X_test_final_seq.shape, y_test_final_seq.shape)

In [ ]:
y_test_pred_s = cnn_gru_model.predict(X_test_seq, verbose=0)

assert y_test_pred_s.shape == y_test_seq.shape, \
    "Kích thước dự báo không khớp với y_test_seq."

if scaler_y is not None:
    y_test_true_trans = scaler_y.inverse_transform(y_test_seq)
    y_test_pred_trans = scaler_y.inverse_transform(y_test_pred_s)
else:
    y_test_true_trans = y_test_seq
    y_test_pred_trans = y_test_pred_s

y_test_true_actual = inverse_target_transform(
    y_test_true_trans, use_log=USE_LOG_TARGET
)
y_test_pred_actual = inverse_target_transform(
    y_test_pred_trans, use_log=USE_LOG_TARGET
)

mae_test = mean_absolute_error(
    y_test_true_actual.ravel(),
    y_test_pred_actual.ravel()
)

rmse_test = np.sqrt(mean_squared_error(
    y_test_true_actual.ravel(),
    y_test_pred_actual.ravel()
))

print(f"MAE trên tập test: {mae_test:.4f}")
print(f"RMSE trên tập test: {rmse_test:.4f}")

assert np.isfinite(y_test_pred_actual).all(), \
    "Giá trị dự báo chứa số không hữu hạn (NaN hoặc Inf)."

In [ ]:
assert "best_obj" in globals(), \
    "Không tìm thấy best_obj. Hãy chạy cell random search trước."

assert "best_config" in globals(), \
    "Không tìm thấy best_config. Hãy chạy cell random search trước."

# Xây dựng lại mô hình tốt nhất từ giai đoạn validation
# để kiểm tra hiện tượng overfitting trên train/val/test.
tf.keras.backend.clear_session()

model_overfit_check = build_cnn_gru_from_cfg(
    best_config,
    (best_obj["X_train_seq"].shape[1], best_obj["X_train_seq"].shape[2])
)
model_overfit_check.set_weights(best_obj["weights"])

y_train_pred_s = model_overfit_check.predict(best_obj["X_train_seq"], verbose=0)
y_val_pred_s   = model_overfit_check.predict(best_obj["X_val_seq"], verbose=0)
y_test_pred_s  = model_overfit_check.predict(best_obj["X_test_seq"], verbose=0)

# Tạo lại scaler target chỉ dùng train (giống giai đoạn validation)
_, _, _, scaler_y_search = scale_data_without_leakage(y_train, y_val, y_test)

y_train_true_trans = scaler_y_search.inverse_transform(best_obj["y_train_seq"])
y_train_pred_trans = scaler_y_search.inverse_transform(y_train_pred_s)
y_val_true_trans   = scaler_y_search.inverse_transform(best_obj["y_val_seq"])
y_val_pred_trans   = scaler_y_search.inverse_transform(y_val_pred_s)
y_test_true_trans  = scaler_y_search.inverse_transform(best_obj["y_test_seq"])
y_test_pred_trans  = scaler_y_search.inverse_transform(y_test_pred_s)

y_train_true = inverse_target_transform(y_train_true_trans, use_log=USE_LOG_TARGET)
y_train_pred = inverse_target_transform(y_train_pred_trans, use_log=USE_LOG_TARGET)
y_val_true   = inverse_target_transform(y_val_true_trans, use_log=USE_LOG_TARGET)
y_val_pred   = inverse_target_transform(y_val_pred_trans, use_log=USE_LOG_TARGET)
y_test_true  = inverse_target_transform(y_test_true_trans, use_log=USE_LOG_TARGET)
y_test_pred  = inverse_target_transform(y_test_pred_trans, use_log=USE_LOG_TARGET)

metrics_rows = []

for split_name, y_t, y_p in [
    ("train", y_train_true, y_train_pred),
    ("val",   y_val_true,   y_val_pred),
    ("test",  y_test_true,  y_test_pred),
]:
    mae  = mean_absolute_error(y_t.ravel(), y_p.ravel())
    rmse = np.sqrt(mean_squared_error(y_t.ravel(), y_p.ravel()))
    metrics_rows.append({"Tập dữ liệu": split_name, "MAE": mae, "RMSE": rmse})

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

# ===== Biểu đồ cột MAE và RMSE =====
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(metrics_df["Tập dữ liệu"], metrics_df["RMSE"],
            color=["#4e79a7", "#f28e2b", "#e15759"])
axes[0].set_title("RMSE theo từng tập dữ liệu")
axes[0].set_ylabel("RMSE")

axes[1].bar(metrics_df["Tập dữ liệu"], metrics_df["MAE"],
            color=["#4e79a7", "#f28e2b", "#e15759"])
axes[1].set_title("MAE theo từng tập dữ liệu")
axes[1].set_ylabel("MAE")

plt.tight_layout()
plt.show()

# ===== Vẽ actual vs predicted =====
n_plot = min(300, len(y_test_true))

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

for ax, title, y_t, y_p in [
    (axes[0], "Train",      y_train_true, y_train_pred),
    (axes[1], "Validation", y_val_true,   y_val_pred),
    (axes[2], "Test",       y_test_true,  y_test_pred),
]:
    n = min(n_plot, len(y_t))
    ax.plot(y_t[:n], label="Thực tế", linewidth=1.5)
    ax.plot(y_p[:n], label="Dự báo", linewidth=1.5)
    ax.set_title(f"{title}: {n} điểm đầu tiên")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ===== Kiểm tra độ lệch RMSE =====
train_rmse = metrics_df.loc[metrics_df["Tập dữ liệu"] == "train", "RMSE"].iloc[0]
val_rmse   = metrics_df.loc[metrics_df["Tập dữ liệu"] == "val",   "RMSE"].iloc[0]
test_rmse  = metrics_df.loc[metrics_df["Tập dữ liệu"] == "test",  "RMSE"].iloc[0]

print(f"Chênh lệch RMSE val/train: {(val_rmse / train_rmse - 1) * 100:.2f}%")
print(f"Chênh lệch RMSE test/train: {(test_rmse / train_rmse - 1) * 100:.2f}%")

In [ ]:
print("Số lượng mẫu (trước khi tạo sequence):")
print("Train/Val/Test:", len(X_train), len(X_val), len(X_test))

print("Số lượng mẫu (sau khi tạo sequence):")
print("Train/Val/Test:", len(X_train_seq), len(X_val_seq), len(X_test_seq))

non_finite_total = (
    np.size(X_train_s) - np.isfinite(X_train_s).sum()
    + np.size(X_val_s) - np.isfinite(X_val_s).sum()
    + np.size(X_test_s) - np.isfinite(X_test_s).sum()
    + np.size(y_train_s) - np.isfinite(y_train_s).sum()
    + np.size(y_val_s) - np.isfinite(y_val_s).sum()
    + np.size(y_test_s) - np.isfinite(y_test_s).sum()
)

print("Tổng số giá trị không hữu hạn (NaN/Inf) trên các mảng đã scale:", int(non_finite_total))
assert non_finite_total == 0, "Phát hiện giá trị không hữu hạn (NaN/Inf) sau khi chuẩn hóa."

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def build_eval_df(y_true, y_pred, idx):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    df_eval = pd.DataFrame({
        "actual": y_true,
        "pred": y_pred
    }, index=pd.to_datetime(idx))

    df_eval["error"] = df_eval["pred"] - df_eval["actual"]
    df_eval["abs_error"] = df_eval["error"].abs()
    df_eval["sq_error"] = df_eval["error"] ** 2
    return df_eval

def calc_metrics(df_eval):
    y_true = df_eval["actual"].values
    y_pred = df_eval["pred"].values
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


lb = best_config["lookback"] if "best_config" in globals() else LOOKBACK

val_idx = val_df.index[lb: lb + len(y_val_true)]
test_idx = test_df.index[lb: lb + len(y_test_true)]

val_eval = build_eval_df(y_val_true, y_val_pred, val_idx)
test_eval = build_eval_df(y_test_true, y_test_pred, test_idx)

metrics_df = pd.DataFrame([
    {"split": "val", **calc_metrics(val_eval)},
    {"split": "test", **calc_metrics(test_eval)},
])
display(metrics_df)


In [ ]:
def plot_error_timeline(df_eval, split_name, rolling_window=24):
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    axes[0].plot(df_eval.index, df_eval["actual"], label="Actual", linewidth=1.2)
    axes[0].plot(df_eval.index, df_eval["pred"], label="Pred", linewidth=1.2, alpha=0.9)
    axes[0].set_title(f"{split_name.upper()} - Actual vs Pred")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_eval.index, df_eval["abs_error"], label="|Error|", alpha=0.5)
    axes[1].plot(
        df_eval.index,
        df_eval["abs_error"].rolling(rolling_window).mean(),
        label=f"Rolling mean |Error| ({rolling_window} points)",
        linewidth=2
    )
    axes[1].set_title(f"{split_name.upper()} - Error over time")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_error_timeline(val_eval, "val", rolling_window=24)
plot_error_timeline(test_eval, "test", rolling_window=24)


print("VAL monthly MAE")
display(val_eval["abs_error"].resample("M").mean().to_frame("MAE_monthly"))

print("TEST monthly MAE")
display(test_eval["abs_error"].resample("M").mean().to_frame("MAE_monthly"))


In [ ]:
# Naive baseline (persistence): y_hat(t) = y(t-1)
assert "val_eval" in globals() and "test_eval" in globals(), "Run evaluation cells first to create val_eval/test_eval."

lb = best_config["lookback"] if "best_config" in globals() else LOOKBACK


def build_naive_eval(split_df, target_col, lookback, n_points):
    series = split_df[target_col].astype(float).values
    max_n = len(series) - lookback
    n = min(n_points, max_n)
    assert n > 0, "Not enough samples to build naive baseline."

    y_true = series[lookback:lookback + n]
    y_pred = series[lookback - 1:lookback - 1 + n]
    idx = split_df.index[lookback:lookback + n]
    return build_eval_df(y_true, y_pred, idx)


val_naive_eval = build_naive_eval(val_df, target, lb, len(val_eval))
test_naive_eval = build_naive_eval(test_df, target, lb, len(test_eval))

compare_df = pd.DataFrame([
    {"split": "val", "model": "CNN-GRU", **calc_metrics(val_eval)},
    {"split": "val", "model": "Naive(t-1)", **calc_metrics(val_naive_eval)},
    {"split": "test", "model": "CNN-GRU", **calc_metrics(test_eval)},
    {"split": "test", "model": "Naive(t-1)", **calc_metrics(test_naive_eval)},
])

compare_df = compare_df[["split", "model", "RMSE", "MAE", "R2"]]
display(compare_df)

for split_name, model_eval, naive_eval in [
    ("val", val_eval, val_naive_eval),
    ("test", test_eval, test_naive_eval),
]:
    m = calc_metrics(model_eval)
    n = calc_metrics(naive_eval)
    rmse_gain = (n["RMSE"] - m["RMSE"]) / n["RMSE"] * 100
    mae_gain = (n["MAE"] - m["MAE"]) / n["MAE"] * 100
    print(f"{split_name.upper()} improvement vs naive -> RMSE: {rmse_gain:.2f}% | MAE: {mae_gain:.2f}%")